<a href="https://colab.research.google.com/github/Swayam17o5/DL1/blob/main/Encooder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Encoder-Decoder LSTM Model
This notebook demonstrates how to build a basic Sequence-to-Sequence (Seq2Seq) Encoder-Decoder model using Keras LSTM layers.

In [4]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

# -----------------------------
# PARAMETERS
# -----------------------------
num_encoder_tokens = 10
num_decoder_tokens = 10
latent_dim = 64
num_samples = 1000
max_seq_length = 5

# -----------------------------
# DUMMY DATASET (Sequence → Sequence)
# Example: input = [1,2,3] → output = [3,2,1]
# -----------------------------
encoder_input_data = np.zeros((num_samples, max_seq_length, num_encoder_tokens))
decoder_input_data = np.zeros((num_samples, max_seq_length, num_decoder_tokens))
decoder_target_data = np.zeros((num_samples, max_seq_length, num_decoder_tokens))

for i in range(num_samples):
    seq = np.random.randint(1, num_encoder_tokens, size=max_seq_length)
    reversed_seq = seq[::-1]

    for t in range(max_seq_length):
        encoder_input_data[i, t, seq[t]] = 1

        # Decoder input (teacher forcing)
        if t == 0:
            decoder_input_data[i, t, 0] = 1  # start token
        else:
            decoder_input_data[i, t, reversed_seq[t-1]] = 1

        # Target output
        decoder_target_data[i, t, reversed_seq[t]] = 1

# -----------------------------
# ENCODER
# -----------------------------
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]

# -----------------------------
# DECODER
# -----------------------------
decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# -----------------------------
# MODEL
# -----------------------------
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

# -----------------------------
# TRAINING
# -----------------------------
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=10,
    validation_split=0.2
)

# -----------------------------
# INFERENCE MODELS
# -----------------------------

# Encoder model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder model
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_inputs,
    initial_state=decoder_states_inputs
)

decoder_states = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states
)

# -----------------------------
# PREDICTION FUNCTION
# -----------------------------
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)

    target_seq = np.zeros((1, 1, num_decoder_tokens))
    target_seq[0, 0, 0] = 1  # start token

    decoded_sequence = []

    for _ in range(max_seq_length):
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        decoded_sequence.append(sampled_token_index)

        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1

        states_value = [h, c]

    return decoded_sequence

# -----------------------------
# TEST
# -----------------------------
test_seq = encoder_input_data[0:1]
print("Predicted sequence:", decode_sequence(test_seq))

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 10)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None, 10)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 64),      │     19,200 │ input_layer[0][0] │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     19,200 │ input_layer_1[0]… │
│                     │ 64), (None, 64),  │            │ lstm[0][1],       │
│                     │ (None, 64)]       │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 10)  │        650 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 39,050 (152.54 KB)

 Trainable params: 39,050 (152.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 48ms/step - accuracy: 0.1470 - loss: 2.2871 - val_accuracy: 0.2070 - val_loss: 2.2627
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2427 - loss: 2.2371 - val_accuracy: 0.2740 - val_loss: 2.2066
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2840 - loss: 2.1635 - val_accuracy: 0.2960 - val_loss: 2.1117
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3040 - loss: 2.0242 - val_accuracy: 0.3110 - val_loss: 1.9263
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3283 - loss: 1.8278 - val_accuracy: 0.3500 - val_loss: 1.7572
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3527 - loss: 1.6905 - val_accuracy: 0.3670 - val_loss: 1.6423
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3817 - loss: 1.5757 - val_accuracy: 0.3820 - val_loss: 1.5397
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3970 - loss: 1.4953 - val_accuracy: 0.3940 - v

In [ ]:
The Encoder–Decoder architecture is a type of sequence-to-sequence model used for predicting output sequences from input sequences. It is commonly implemented using Long Short-Term Memory (LSTM) networks.

The model consists of two main parts:

🔹 Encoder

The encoder takes the input sequence and processes it step-by-step using an LSTM layer. It converts the entire input sequence into fixed-length context vectors, namely:

Hidden state (h)
Cell state (c)

These states capture the important information from the input sequence and are passed to the decoder.

🔹 Decoder

The decoder is another LSTM network that generates the output sequence. It is initialized with the encoder’s final hidden and cell states.

At each time step:

The decoder takes the previous output as input
Produces the next output in the sequence

During training, teacher forcing is used, where the correct previous output is given as input to the decoder.

🔹 Output Layer

A Dense layer with **Softmax function activation is used to convert decoder outputs into probability distributions over possible tokens.

🔹 Training

The model is trained using:

Loss function: Categorical Crossentropy
Optimizer: Adam/RMSprop

The goal is to minimize the difference between predicted and actual sequences.

🔹 Applications
Machine translation
Chatbots
Text summarization
Time series prediction